## Connexion à PostgreSQL

In [ ]:
# 1 : importation 
from dotenv import load_dotenv
import os
import psycopg2
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@"
    f"{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}",
    echo=False,
    pool_pre_ping=True
)

## Création des tables

In [28]:
from sqlalchemy import create_engine, text

sql = """
-- Table des agences
CREATE TABLE IF NOT EXISTS agences (
    agence_id SERIAL PRIMARY KEY,
    agence VARCHAR(100) UNIQUE
);

-- Table des segments clients
CREATE TABLE IF NOT EXISTS segments_client (
    segment_id SERIAL PRIMARY KEY,
    segment_client VARCHAR(50) UNIQUE,
    categorie_risque VARCHAR(50)
);

-- Table des clients
CREATE TABLE IF NOT EXISTS clients (
    client_id VARCHAR PRIMARY KEY,
    segment_client VARCHAR(50),
    score_credit_client NUMERIC(10,2)
);

-- Table des produits
CREATE TABLE IF NOT EXISTS produits (
    produit_id SERIAL PRIMARY KEY,
    produit VARCHAR(100) UNIQUE
);

-- Table des transactions
CREATE TABLE IF NOT EXISTS transactions (
    transaction_id BIGINT PRIMARY KEY,
    client_id VARCHAR,
    date_transaction DATE,
    montant NUMERIC(15,2),
    devise VARCHAR(10),
    taux_change_eur NUMERIC(10,4),
    montant_eur NUMERIC(15,2),
    categorie VARCHAR(100),
    produit VARCHAR(100),
    agence VARCHAR(100),
    type_operation VARCHAR(50),
    statut VARCHAR(50),
    score_credit_client NUMERIC(10,2),
    segment_client VARCHAR(50),
    solde_avant NUMERIC(15,2),
    anomaly_montant NUMERIC(10,2),
    anomaly_score NUMERIC(10,2),
    is_anomaly BOOLEAN,
    annee INT,
    mois INT,
    trimestre INT,
    jour_semaine VARCHAR(20),
    montant_eur_verifie NUMERIC(15,2),
    comparaison VARCHAR(50),
    categorie_risque VARCHAR(50),
    taux_rejet NUMERIC(10,2),
    CONSTRAINT fk_transactions_clients
        FOREIGN KEY (client_id) REFERENCES clients(client_id)
);
"""

with engine.begin() as connection:
    connection.execute(text(sql))

print("✅ Tables créées avec succès")

✅ Tables créées avec succès


In [29]:
import pandas as pd

df = pd.read_csv('financecore_clean.csv')
df

,transaction_id,client_id,date_transaction,montant,devise,taux_change_eur,montant_eur,categorie,produit,agence,...,is_anomaly,annees,mois,trimestre,jour_semaine,montant_eur_verifie,ecart_eur,is_conversion_error,categorie_risque,taux_rejet
0,TXN000559,CLI0060,2022-04-19 02:31:00,2050.42,EUR,1.00,2050.42,Depot especes,Compte Epargne,Marseille-Vieux-Port,...,False,2022,4,2,1,2050.420000,0.000000,False,Medium,5.405405
1,TXN001154,CLI0057,2024-06-20 20:51:00,-123.66,GBP,0.86,-143.79,Retrait DAB,Credit Consommation,Bordeaux-Meriadeck,...,False,2024,6,2,3,-143.790698,0.000698,False,High,4.428904
2,TXN000764,CLI0015,2024-08-28 05:03:00,-396.17,EUR,1.00,-396.17,Prelevement,PEA,Lyon-Part-Dieu,...,False,2024,8,3,2,-396.170000,0.000000,False,Medium,4.193548
3,TXN001598,CLI0045,2024-01-07 08:16:00,225.20,EUR,1.00,225.20,Paiement CB,Credit Consommation,Bordeaux-Meriadeck,...,False,2024,1,1,6,225.200000,0.000000,False,Low,4.428904
4,TXN001873,CLI0034,2024-08-11 19:52:00,935.32,EUR,1.00,935.32,Interets,Credit Immobilier,Bordeaux-Meriadeck,...,False,2024,8,3,6,935.320000,0.000000,False,High,4.428904
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2055,TXN001093,CLI0044,2022-11-18 10:33:00,2944.41,EUR,1.00,2944.41,Depot especes,Compte Epargne,Lyon-Part-Dieu,...,False,2022,11,4,4,2944.410000,0.000000,False,High,4.193548
2056,TXN001769,CLI0040,2024-03-28 11:34:00,296.96,EUR,1.00,296.96,Virement,Credit Immobilier,Bordeaux-Meriadeck,...,False,2024,3,1,3,296.960000,0.000000,False,High,4.428904
2057,TXN001738,CLI0106,2023-07-05 15:28:30,-2106.58,EUR,1.00,-2106.58,Virement international,Assurance Vie,Nantes-Commerce,...,False,2023,7,3,2,-2106.580000,0.000000,False,Medium,8.095238
2058,TXN001210,CLI0098,2024-02-18 12:02:00,-353.74,CHF,0.97,-364.68,Retrait DAB,Compte Epargne,Bordeaux-Meriadeck,...,False,2024,2,1,6,-364.680412,0.000412,False,Medium,4.428904


In [30]:
df.columns

Index(['transaction_id', 'client_id', 'date_transaction', 'montant', 'devise',
       'taux_change_eur', 'montant_eur', 'categorie', 'produit', 'agence',
       'type_operation', 'statut', 'score_credit_client', 'segment_client',
       'solde_avant', 'taux_interet', 'anomaly_montant', 'anomaly_score',
       'is_anomaly', 'annees', 'mois', 'trimestre', 'jour_semaine',
       'montant_eur_verifie', 'ecart_eur', 'is_conversion_error',
       'categorie_risque', 'taux_rejet'],
      dtype='object')

In [31]:
agences_df = df["agence"].drop_duplicates()
segments_clients_df = df[["segment_client", "categorie_risque"]].drop_duplicates()
clients_df = df[["client_id", "segment_client", "score_credit_client"]].drop_duplicates()
produits_df = df[["produit"]].drop_duplicates()
transactions_df = df[[
    "transaction_id",
    "client_id",
    "date_transaction",
    "montant",
    "devise",
    "taux_change_eur",
    "montant_eur",
    "categorie",
    "produit",
    "agence",
    "type_operation",
    "statut",
    "solde_avant",
    "taux_interet",
    "is_anomaly",
    "annees",
    "mois",
    "trimestre",
    "jour_semaine",
    "montant_eur_verifie",
    "ecart_eur",
    "taux_rejet"
]].drop_duplicates(subset="transaction_id")

#  Chargement des Données via SQLAlchemy

In [33]:
from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(text("DROP VIEW IF EXISTS vw_kpi_taux_defaut;"))

In [35]:
agences_df.to_sql("agences", engine, if_exists="replace", index=False)
segments_clients_df.to_sql("segments_client", engine, if_exists="replace", index=False)
produits_df.to_sql("produits", engine, if_exists="replace", index=False)
segments_clients_df.to_sql("segments_client", engine, if_exists="replace", index=False)

11

In [36]:
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS clients CASCADE"))
clients_df.to_sql("clients", engine, if_exists="replace", index=False)

333

In [ ]:
with engine.begin() as conn:
    result = conn.execute(text('''
        SELECT
            agence,
            produit,
            annees,
            mois,
            COUNT(*) AS nombre_transactions,
            SUM(montant_eur) AS total_montant_eur,
            AVG(montant_eur) AS moyenne_montant_eur
        FROM
            transactions
        GROUP BY
            agence,
            produit,
            annees,
            mois
        HAVING
            SUM(montant_eur) > 1000
        ORDER BY
            total_montant_eur DESC
        LIMIT 10;
    '''))

rows = result.fetchall()

for row in rows:
    print(dict(row._mapping))  
    print(dict(row._mapping))

{'agence': 'Toulouse-Capitole', 'produit': 'Compte Epargne', 'annees': 2024, 'mois': 11, 'nombre_transactions': 2, 'total_montant_eur': 10260.52, 'moyenne_montant_eur': 5130.26}
{'agence': 'Toulouse-Capitole', 'produit': 'Compte Epargne', 'annees': 2024, 'mois': 11, 'nombre_transactions': 2, 'total_montant_eur': 10260.52, 'moyenne_montant_eur': 5130.26}
{'agence': 'Lyon-Part-Dieu', 'produit': 'Credit Immobilier', 'annees': 2022, 'mois': 12, 'nombre_transactions': 2, 'total_montant_eur': 10026.789999999999, 'moyenne_montant_eur': 5013.3949999999995}
{'agence': 'Lyon-Part-Dieu', 'produit': 'Credit Immobilier', 'annees': 2022, 'mois': 12, 'nombre_transactions': 2, 'total_montant_eur': 10026.789999999999, 'moyenne_montant_eur': 5013.3949999999995}
{'agence': 'Lyon-Part-Dieu', 'produit': 'Compte Courant', 'annees': 2022, 'mois': 5, 'nombre_transactions': 3, 'total_montant_eur': 9582.220000000001, 'moyenne_montant_eur': 3194.0733333333337}
{'agence': 'Lyon-Part-Dieu', 'produit': 'Compte Cour

In [ ]:
with engine.begin() as conn:
        result = conn.execute(text('''
        SELECT
            client_id,
            solde_avant
        FROM
            transactions
        WHERE
            solde_avant < (
                SELECT AVG(solde_avant)
                FROM transactions
            )
        ORDER BY
            solde_avant
        LIMIT 10;
'''))

rows = result.fetchall()

for row in rows:
    print(dict(row._mapping))

{'client_id': 'CLI0082', 'solde_avant': 103.04}
{'client_id': 'CLI0003', 'solde_avant': 112.31}
{'client_id': 'CLI0016', 'solde_avant': 127.18}
{'client_id': 'CLI0101', 'solde_avant': 179.12}
{'client_id': 'CLI0149', 'solde_avant': 212.3}
{'client_id': 'CLI0075', 'solde_avant': 222.09}
{'client_id': 'CLI0072', 'solde_avant': 260.32}
{'client_id': 'CLI0133', 'solde_avant': 267.6}
{'client_id': 'CLI0117', 'solde_avant': 308.61}
{'client_id': 'CLI0118', 'solde_avant': 335.2}


In [ ]:
with engine.begin() as conn:
    result = conn.execute(text('''
        SELECT
            sc.categorie_risque,
            COUNT(*) AS nombre_total_transactions,
            COUNT(*) FILTER (
                WHERE t.statut IN ('Rejeté', 'Refusé', 'Défaut')
            ) AS nombre_defauts,
            ROUND(
                100.0 * COUNT(*) FILTER (
                    WHERE t.statut IN ('Rejeté', 'Refusé', 'Défaut')
                ) / NULLIF(COUNT(*), 0),
                2
            ) AS taux_defaut_pourcentage
        FROM transactions t
        JOIN clients c ON t.client_id = c.client_id
        JOIN segments_client sc ON c.segment_client = sc.segment_client
        GROUP BY sc.categorie_risque
        ORDER BY taux_defaut_pourcentage DESC;
    '''))

rows = result.fetchall()

for row in rows:
    print(dict(row._mapping))

{'categorie_risque': 'Medium', 'nombre_total_transactions': 4607, 'nombre_defauts': 0, 'taux_defaut_pourcentage': Decimal('0.00')}
{'categorie_risque': 'High', 'nombre_total_transactions': 2623, 'nombre_defauts': 0, 'taux_defaut_pourcentage': Decimal('0.00')}
{'categorie_risque': 'Low', 'nombre_total_transactions': 4607, 'nombre_defauts': 0, 'taux_defaut_pourcentage': Decimal('0.00')}


In [ ]:
with engine.begin() as conn:
    result = conn.execute(text('''
        SELECT
            t.transaction_id,
            t.date_transaction,
            t.montant_eur,
            c.client_id,
            c.segment_client,
            sc.categorie_risque,
            a.agence,
            p.produit,
            t.statut
        FROM transactions t
        INNER JOIN clients c 
            ON t.client_id = c.client_id
        INNER JOIN segments_client sc 
            ON c.segment_client = sc.segment_client
        INNER JOIN agences a 
            ON t.agence = a.agence
        INNER JOIN produits p 
            ON t.produit = p.produit
        ORDER BY t.date_transaction DESC
        LIMIT 10;
    '''))

rows = result.fetchall()

for row in rows:
    print(dict(row._mapping))

{'transaction_id': 'TXN000850', 'date_transaction': '2024-12-31 14:06:00', 'montant_eur': 869.42, 'client_id': 'CLI0024', 'segment_client': 'Standard', 'categorie_risque': 'Medium', 'agence': 'Lyon-Part-Dieu', 'produit': 'Compte Epargne', 'statut': 'Complete'}
{'transaction_id': 'TXN000850', 'date_transaction': '2024-12-31 14:06:00', 'montant_eur': 869.42, 'client_id': 'CLI0024', 'segment_client': 'Standard', 'categorie_risque': 'Low', 'agence': 'Lyon-Part-Dieu', 'produit': 'Compte Epargne', 'statut': 'Complete'}
{'transaction_id': 'TXN001516', 'date_transaction': '2024-12-31 12:59:00', 'montant_eur': 674.87, 'client_id': 'CLI0096', 'segment_client': 'Standard', 'categorie_risque': 'Medium', 'agence': 'Lyon-Part-Dieu', 'produit': 'Compte Courant', 'statut': 'Complete'}
{'transaction_id': 'TXN001516', 'date_transaction': '2024-12-31 12:59:00', 'montant_eur': 674.87, 'client_id': 'CLI0096', 'segment_client': 'Nan', 'categorie_risque': 'Medium', 'agence': 'Lyon-Part-Dieu', 'produit': 'Com

In [ ]:
with engine.begin() as conn:
    conn.execute(text('''
        CREATE OR REPLACE VIEW vw_kpi_transactions AS
        SELECT
            t.agence,
            t.produit,
            t.annees,
            t.mois,
            COUNT(*) AS nombre_transactions,
            SUM(t.montant_eur) AS montant_total_eur,
            AVG(t.montant_eur) AS montant_moyen_eur
        FROM transactions t
        GROUP BY
            t.agence,
            t.produit,
            t.annees,
            t.mois;
    '''))

In [ ]:
with engine.begin() as conn:
    conn.execute(text('''
        CREATE OR REPLACE VIEW vw_kpi_taux_defaut AS
        SELECT
            sc.categorie_risque,
            COUNT(*) AS nombre_transactions,
            COUNT(*) FILTER (
                WHERE t.statut IN ('Rejeté', 'Refusé', 'Défaut')
            ) AS nombre_defauts,
            ROUND(
                100.0 * COUNT(*) FILTER (
                    WHERE t.statut IN ('Rejeté', 'Refusé', 'Défaut')
                ) / NULLIF(COUNT(*), 0),
                2
            ) AS taux_defaut_pourcentage
        FROM transactions t
        JOIN clients c ON t.client_id = c.client_id
        JOIN segments_client sc ON c.segment_client = sc.segment_client
        GROUP BY sc.categorie_risque;
    '''))

In [ ]:
with engine.begin() as conn:
    conn.execute(text('''
        CREATE OR REPLACE VIEW vw_kpi_performance_agence AS
        SELECT
            t.agence,
            COUNT(*) AS nombre_transactions,
            SUM(t.montant_eur) AS montant_total,
            AVG(t.montant_eur) AS montant_moyen,
            COUNT(DISTINCT t.client_id) AS nombre_clients_uniques
        FROM transactions t
        GROUP BY t.agence;
    '''))